# 01 – Data Acquisition

**Stage 2 goal:** confirm that the raw Amazon dataset is in place, unchanged, and loads correctly.

No cleaning happens here. We only *look* at the data. Cleaning is Stage 3.

The reusable logic lives in `src/amazon_project/loading.py`; this notebook just imports and uses it.

## 1. Imports
`hashlib` computes file checksums (a fingerprint of a file's bytes). `load_raw` and `RAW_DATA_PATH` come from our own package.

In [ ]:
import hashlib

from amazon_project.loading import RAW_DATA_PATH, load_raw

## 2. Verify the raw file is unchanged
We compute the file's SHA-256 checksum and compare it to the one recorded in the README. If even one byte changed, the checksum would be completely different.

In [ ]:
EXPECTED_SHA256 = "4ba126c4ba8edd35e62e94ce1c853073d832de380184ab4f5f97d1e314882c77"

actual_sha256 = hashlib.sha256(RAW_DATA_PATH.read_bytes()).hexdigest()
print("File:    ", RAW_DATA_PATH.name)
print("Size:    ", f"{RAW_DATA_PATH.stat().st_size:,} bytes")
print("Checksum matches README:", actual_sha256 == EXPECTED_SHA256)

## 3. Load the data
`load_raw()` reads every column as plain text, exactly as stored (so `₹1,099` stays `₹1,099`).

In [ ]:
df = load_raw()
df.shape

The shape is `(rows, columns)`. We expect 1,465 rows and 16 columns.

## 4. First look

In [ ]:
df.head(3)

In [ ]:
df.columns.tolist()

`.info()` lists each column's type and non-null count. Every column is text (`str`) because we loaded it untouched. Turning prices, ratings, etc. into numbers is a Stage 3 task.

In [ ]:
df.info()

## 5. Empty values
Because we load without interpreting anything, missing values appear as **empty strings** rather than `NaN`. Counting them shows where the gaps are.

In [ ]:
(df == "").sum()[lambda s: s > 0]

## 6. Summary
- Raw file present, checksum verified.
- Loads as 1,465 rows × 16 text columns.
- Known issues (text-formatted numbers, empty `rating_count`, a `|` rating, repeated `product_id`s, comma-packed review columns) are documented in `CLAUDE.md` and will be handled in Stage 3.